🎯 Goal of Phase 3

Transform clean Silver data into AI-ready features that help models:

- Understand demand patterns  
- Detect trends  
- Anticipate stock-outs

Table: inventory_ai.gold_demand_features

-Featured columns:

- date
- store_id
- product_family
- daily_sales
- onpromotion_count
- avg_7d_sales
- avg_14d_sales
- sales_trend
- promo_ratio


Loading Silver Table

In [0]:

spark.sql("USE inventory_ai")

silver_df = spark.table("silver_daily_sales")
silver_df.show(5)


Define time window

In [0]:
from pyspark.sql.window import Window

sales_window_7d = Window.partitionBy(
    "store_id", "product_family"
).orderBy("date").rowsBetween(-6, 0)

sales_window_14d = Window.partitionBy(
    "store_id", "product_family"
).orderBy("date").rowsBetween(-13, 0)


In [0]:
from pyspark.sql.functions import avg,round

features_df = (
    silver_df
    .withColumn("avg_7d_sales", round(avg("daily_sales").over(sales_window_7d),2))
    .withColumn("avg_14d_sales",round(avg("daily_sales").over(sales_window_14d),2))
)


In [0]:
display(features_df.limit(10))

Sales Trend

In [0]:
from pyspark.sql.functions import col

features_df = features_df.withColumn(
    "sales_trend",
    round(col("avg_7d_sales") - col("avg_14d_sales"),4)
)


In [0]:
display(features_df.limit(10))

In [0]:
features_df = features_df.withColumn(
    "promo_ratio",
    col("onpromotion_count") / (col("avg_7d_sales") + 1)
)


### ## Handle NULLs from Rolling Windows

In [0]:
features_df = features_df.fillna({
    "avg_7d_sales": 0,
    "avg_14d_sales": 0,
    "sales_trend": 0,
    "promo_ratio": 0
})


In [0]:
features_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("inventory_ai.gold_demand_features")


In [0]:
spark.sql("""
SELECT *
FROM gold_demand_features
ORDER BY date
LIMIT 10
""").display()
